In [7]:
%%writefile /kaggle/working/config.py
import torch

# ==========================================================
# Paths
# ==========================================================

PNG_DIR = "/kaggle/input/datasets/hareemzia58/rsna-intracranial-hemorrhage-png-384x384/rsna_dataset_png/rsna_dataset_png"

METADATA_CSV = "/kaggle/input/datasets/hareemzia58/rsna-intracranial-hemorrhage-png-384x384/rsna_metadata.csv"

CHECKPOINT_DIR = "/kaggle/working/checkpoints"

RESULTS_DIR = "/kaggle/working/results"

# ==========================================================
# Data
# ==========================================================

IMG_SIZE = 384

BATCH_SIZE = 16

NUM_WORKERS = 2

PATIENT_ID_COL = "PatientID"

IMAGE_ID_COL = "ImageID"

# ==========================================================
# Labels
# ==========================================================

LABEL_COLS = [
    "any",
    "epidural",
    "intraparenchymal",
    "intraventricular",
    "subarachnoid",
    "subdural",
]

NUM_CLASSES = len(LABEL_COLS)

# ==========================================================
# Model
# ==========================================================

MODEL_VARIANT = "efficientnet_b0"

PRETRAINED = True

DROPOUT = 0.3

# ==========================================================
# Optimizer
# ==========================================================

LR = 2e-4

WEIGHT_DECAY = 1e-4

# ==========================================================
# Class-Balanced Focal Loss
# ==========================================================

CB_BETA = 0.9999

FOCAL_GAMMA = 2.0

# ==========================================================
# Training
# ==========================================================

EPOCHS = 50

PATIENCE = 8

USE_AMP = True

SEED = 42

# ==========================================================
# Device
# ==========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

Overwriting /kaggle/working/config.py


In [8]:
%%writefile /kaggle/working/model.py
import torch.nn as nn
from torchvision import models


_MODEL_REGISTRY = {
    "efficientnet_b0": (
        models.efficientnet_b0,
        models.EfficientNet_B0_Weights.IMAGENET1K_V1,
    )
}


class HemorrhageEfficientNet(nn.Module):

    def __init__(
        self,
        num_classes=6,
        pretrained=True,
        dropout=0.3,
        variant="efficientnet_b0",
    ):

        super().__init__()

        constructor, weights = _MODEL_REGISTRY[variant]

        self.backbone = constructor(
            weights=weights if pretrained else None
        )

        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Sequential(

            nn.Dropout(dropout),

            nn.Linear(
                in_features,
                num_classes,
            ),
        )

    def forward(self, x):

        return self.backbone(x)


def build_model(device=None):

    model = HemorrhageEfficientNet(
        num_classes=6,
        pretrained=True,
        dropout=0.3,
        variant="efficientnet_b0",
    )

    if device is not None:
        model = model.to(device)

    return model

Overwriting /kaggle/working/model.py


In [9]:
%%writefile /kaggle/working/losses.py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class ClassBalancedFocalLoss(nn.Module):
    """
    Class-Balanced Focal Loss
    Cui et al. (CVPR 2019)

    Uses the Effective Number of Samples:
        weight_i = (1-beta)/(1-beta^n_i)
    where n_i is the number of positive samples for class i.
    """

    def __init__(
        self,
        samples_per_class,
        beta=0.9999,
        gamma=2.0,
        reduction="mean",
    ):
        super().__init__()

        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / effective_num
        weights = weights / weights.mean()

        self.register_buffer(
            "class_weights",
            torch.tensor(weights, dtype=torch.float32),
        )
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )

        probs = torch.sigmoid(logits)
        pt = probs * targets + (1 - probs) * (1 - targets)
        focal = (1 - pt) ** self.gamma

        class_weights = self.class_weights.to(logits.device)
        alpha_factor = targets * class_weights + (1.0 - targets)

        loss = alpha_factor * focal * bce

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()

        return loss

Overwriting /kaggle/working/losses.py


In [10]:
%%writefile /kaggle/working/data_loader.py
import os
import cv2
import torch
import numpy as np
import pandas as pd

from torch.utils.data import (
    Dataset,
    DataLoader,
)

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import GroupKFold

# -------------------------------------------------------
# Augmentations
# -------------------------------------------------------

def get_train_transforms(img_size=384):

    return A.Compose([

        A.HorizontalFlip(p=0.5),

        A.Affine(
            translate_percent=(-0.03, 0.03),
            scale=(0.95, 1.05),
            rotate=(-10, 10),
            p=0.5,
        ),

        A.CLAHE(
            clip_limit=(1,4),
            tile_grid_size=(8,8),
            p=0.30,
        ),

        A.RandomGamma(
            gamma_limit=(85,115),
            p=0.30,
        ),

        A.GaussNoise(
            std_range=(0.01,0.03),
            mean_range=(0.0,0.0),
            p=0.20,
        ),

        A.Resize(img_size, img_size),

        A.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225],
        ),

        ToTensorV2(),

    ])


def get_val_transforms(img_size=384):

    return A.Compose([

        A.Resize(img_size, img_size),

        A.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225],
        ),

        ToTensorV2(),

    ])


# -------------------------------------------------------
# Dataset
# -------------------------------------------------------

class RSNADataset(Dataset):

    def __init__(self, df, img_dir, transform=None):

        self.df = df.reset_index(drop=True)

        self.img_dir = img_dir

        self.transform = transform

        self.label_cols = [

            "any",

            "epidural",

            "intraparenchymal",

            "intraventricular",

            "subarachnoid",

            "subdural",

        ]

        self.labels = self.df[self.label_cols].values.astype(np.float32)

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        image_id = self.df.iloc[idx]["Image_ID"]

        path = os.path.join(
            self.img_dir,
            image_id + ".png",
        )

        image = cv2.imread(path)

        if image is None:
            raise FileNotFoundError(path)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:

            image = self.transform(image=image)["image"]

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.float32,
        )

        return image, label


# -------------------------------------------------------
# Data loaders
# -------------------------------------------------------

def prepare_dataloaders(

    df,

    img_dir,

    batch_size,

    num_workers,

    img_size,

    patient_col="PatientID",

):

    groups = df[patient_col]

    y = df["any"]

    outer = StratifiedGroupKFold(

        n_splits=5,

        shuffle=True,

        random_state=42,

    )

    train_idx, temp_idx = next(

        outer.split(df, y, groups)

    )

    train_df = df.iloc[train_idx].reset_index(drop=True)

    temp_df = df.iloc[temp_idx].reset_index(drop=True)

    inner = GroupKFold(n_splits=2)
    val_idx, test_idx = next(
        inner.split(
            temp_df,
            groups=temp_df[patient_col],
        )
    )
    val_df = temp_df.iloc[val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)

    print()

    print("Dataset Summary")

    print("----------------------------")

    print("Train :", len(train_df))

    print("Val   :", len(val_df))

    print("Test  :", len(test_df))

    train_dataset = RSNADataset(

        train_df,

        img_dir,

        get_train_transforms(img_size),

    )

    val_dataset = RSNADataset(

        val_df,

        img_dir,

        get_val_transforms(img_size),

    )

    test_dataset = RSNADataset(

        test_df,

        img_dir,

        get_val_transforms(img_size),

    )

    pin = torch.cuda.is_available()
    
    train_loader = DataLoader(
    
        train_dataset,
    
        batch_size=batch_size,
    
        shuffle=True,
    
        num_workers=num_workers,
    
        pin_memory=pin,
    
        persistent_workers=(num_workers > 0),
    
        prefetch_factor=2 if num_workers > 0 else None,
    
        drop_last=False,
    
    )

    val_loader = DataLoader(

        val_dataset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=num_workers,

        pin_memory=pin,

        persistent_workers=(num_workers > 0),

        prefetch_factor=2 if num_workers > 0 else None,

    )

    test_loader = DataLoader(

        test_dataset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=num_workers,

        pin_memory=pin,

        persistent_workers=(num_workers > 0),

        prefetch_factor=2 if num_workers > 0 else None,

    )

    samples_per_class = train_df[
        [
            "any",
            "epidural",
            "intraparenchymal",
            "intraventricular",
            "subarachnoid",
            "subdural",
        ]
    ].sum().values.astype(np.int64)

    return (
        train_loader,
        val_loader,
        test_loader,
        samples_per_class,
    )

Overwriting /kaggle/working/data_loader.py


In [11]:
%%writefile /kaggle/working/plotting.py
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, confusion_matrix, average_precision_score

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")


# -------------------------------------------------------
# 1. Training vs Validation Loss & Accuracy/F1 Curves
# -------------------------------------------------------
def plot_training_history(history, save_dir):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss Curve
    axes[0].plot(epochs, history["train_loss"], label="Train Loss", color="#1f77b4", linewidth=2)
    axes[0].plot(epochs, history["val_loss"], label="Val Loss", color="#ff7f0e", linewidth=2)
    axes[0].set_title("Training vs Validation Loss", fontsize=14, fontweight="bold")
    axes[0].set_xlabel("Epochs", fontsize=12)
    axes[0].set_ylabel("Loss", fontsize=12)
    axes[0].legend(fontsize=11)
    axes[0].grid(True, linestyle="--", alpha=0.6)

    # Val Macro F1 Curve
    axes[1].plot(epochs, history["val_macro_f1"], label="Val Macro F1", color="#2ca02c", linewidth=2, marker="o")
    axes[1].set_title("Validation Macro F1 Score", fontsize=14, fontweight="bold")
    axes[1].set_xlabel("Epochs", fontsize=12)
    axes[1].set_ylabel("Macro F1", fontsize=12)
    axes[1].legend(fontsize=11)
    axes[1].grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    path = os.path.join(save_dir, "training_history_curves.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


# -------------------------------------------------------
# 2. Combined Precision-Recall Curves
# -------------------------------------------------------
def plot_combined_pr_curves(y_true, probabilities, label_names, save_dir):
    plt.figure(figsize=(10, 7))

    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

    for i, (label, color) in enumerate(zip(label_names, colors)):
        precision, recall, _ = precision_recall_curve(y_true[:, i], probabilities[:, i])
        pr_auc = average_precision_score(y_true[:, i], probabilities[:, i])
        plt.plot(recall, precision, color=color, linewidth=2, label=f"{label} (AUC = {pr_auc:.4f})")

    plt.title("Combined Precision-Recall Curves (Test Set)", fontsize=14, fontweight="bold")
    plt.xlabel("Recall", fontsize=12)
    plt.ylabel("Precision", fontsize=12)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.legend(loc="lower left", fontsize=10)
    plt.grid(True, linestyle="--", alpha=0.6)

    path = os.path.join(save_dir, "combined_pr_curves.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


# -------------------------------------------------------
# 3. F1 Score vs Threshold Curves
# -------------------------------------------------------
def plot_f1_vs_thresholds(y_true, probabilities, label_names, best_thresholds, save_dir):
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    threshold_range = np.linspace(0.01, 0.99, 100)

    for i, label in enumerate(label_names):
        f1_scores = []
        precision, recall, ths = precision_recall_curve(y_true[:, i], probabilities[:, i])

        # Compute F1 for all thresholds
        for th in threshold_range:
            preds = (probabilities[:, i] >= th).astype(int)
            tp = np.sum((preds == 1) & (y_true[:, i] == 1))
            fp = np.sum((preds == 1) & (y_true[:, i] == 0))
            fn = np.sum((preds == 0) & (y_true[:, i] == 1))

            prec = tp / (tp + fp + 1e-12)
            rec = tp / (tp + fn + 1e-12)
            f1 = (2 * prec * rec) / (prec + rec + 1e-12)
            f1_scores.append(f1)

        ax = axes[i]
        ax.plot(threshold_range, f1_scores, color="#1f77b4", linewidth=2)
        ax.axvline(best_thresholds[i], color="#d62728", linestyle="--", linewidth=1.5, label=f"Optimal = {best_thresholds[i]:.3f}")
        ax.set_title(f"F1 vs Threshold: {label}", fontsize=12, fontweight="bold")
        ax.set_xlabel("Threshold", fontsize=10)
        ax.set_ylabel("F1 Score", fontsize=10)
        ax.legend(loc="lower center", fontsize=9)
        ax.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    path = os.path.join(save_dir, "f1_vs_threshold_curves.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


# -------------------------------------------------------
# 4. Multi-Label Confusion Matrices (2x3 Grid)
# -------------------------------------------------------
def plot_confusion_matrices(y_true, y_pred, label_names, save_dir):
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()

    for i, label in enumerate(label_names):
        cm = confusion_matrix(y_true[:, i], y_pred[:, i])
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            ax=axes[i],
            cbar=False,
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"],
            annot_kws={"size": 12, "weight": "bold"},
        )
        axes[i].set_title(f"Confusion Matrix: {label}", fontsize=12, fontweight="bold")
        axes[i].set_xlabel("Predicted Label", fontsize=10)
        axes[i].set_ylabel("True Label", fontsize=10)

    plt.tight_layout()
    path = os.path.join(save_dir, "confusion_matrices.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

Overwriting /kaggle/working/plotting.py


In [12]:
%%writefile /kaggle/working/metrics.py
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    average_precision_score,
    precision_recall_curve,
    f1_score,
)

LABELS = [
    "any",
    "epidural",
    "intraparenchymal",
    "intraventricular",
    "subarachnoid",
    "subdural",
]


def apply_thresholds(probabilities, thresholds):
    thresholds = np.asarray(thresholds)
    return (probabilities >= thresholds).astype(np.uint8)


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)


def compute_pr_auc(y_true, probabilities):
    return {label: average_precision_score(y_true[:, i], probabilities[:, i]) for i, label in enumerate(LABELS)}


def find_best_thresholds(y_true, probabilities):
    best_thresholds = []
    best_f1_scores = []

    for i in range(len(LABELS)):
        precision, recall, thresholds = precision_recall_curve(y_true[:, i], probabilities[:, i])
        f1 = (2 * precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-12)
        best_idx = np.argmax(f1)
        best_thresholds.append(thresholds[best_idx])
        best_f1_scores.append(f1[best_idx])

    return np.array(best_thresholds), np.array(best_f1_scores)


def evaluate_with_thresholds(y_true, probabilities, thresholds, save_dir=None):
    preds = apply_thresholds(probabilities, thresholds)
    macro = macro_f1(y_true, preds)
    report_str = classification_report(y_true, preds, target_names=LABELS, zero_division=0)
    report_dict = classification_report(y_true, preds, target_names=LABELS, zero_division=0, output_dict=True)
    pr_auc = compute_pr_auc(y_true, probabilities)

    # Construct Metrics DataFrame
    summary_rows = []
    for i, label in enumerate(LABELS):
        summary_rows.append({
            "Subtype": label,
            "PR-AUC": pr_auc[label],
            "Threshold": thresholds[i],
            "Precision": report_dict[label]["precision"],
            "Recall": report_dict[label]["recall"],
            "F1-Score": report_dict[label]["f1-score"],
            "Support": report_dict[label]["support"],
        })
    metrics_df = pd.DataFrame(summary_rows)

    # Save outputs if path provided
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        metrics_df.to_csv(os.path.join(save_dir, "summary_metrics.csv"), index=False)
        with open(os.path.join(save_dir, "classification_report.txt"), "w") as f:
            f.write(f"Test Set Macro F1: {macro:.4f}\n\n")
            f.write(report_str)

    return {
        "macro_f1": macro,
        "report_str": report_str,
        "metrics_df": metrics_df,
        "pr_auc": pr_auc,
        "predictions": preds,
    }

Overwriting /kaggle/working/metrics.py


In [14]:
%%writefile /kaggle/working/train.py
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.amp import autocast, GradScaler

import config
from model import build_model
from losses import ClassBalancedFocalLoss
from data_loader import prepare_dataloaders
from metrics import find_best_thresholds, evaluate_with_thresholds
from plotting import (
    plot_training_history,
    plot_combined_pr_curves,
    plot_f1_vs_thresholds,
    plot_confusion_matrices,
)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(config.SEED)

os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(config.RESULTS_DIR, exist_ok=True)

# -------------------------------------------------------
# Data Loader Setup
# -------------------------------------------------------
df = pd.read_csv(config.METADATA_CSV)

train_loader, val_loader, test_loader, samples_per_class = prepare_dataloaders(
    df=df,
    img_dir=config.PNG_DIR,
    batch_size=config.BATCH_SIZE,
    num_workers=config.NUM_WORKERS,
    img_size=config.IMG_SIZE,
    patient_col=config.PATIENT_ID_COL,
)

# -------------------------------------------------------
# Model & Loss Initialization
# -------------------------------------------------------
model = build_model(config.DEVICE)

criterion = ClassBalancedFocalLoss(
    samples_per_class=samples_per_class,
    beta=config.CB_BETA,
    gamma=config.FOCAL_GAMMA,
).to(config.DEVICE)

optimizer = AdamW(model.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
scaler = GradScaler("cuda", enabled=config.USE_AMP)

best_macro_f1 = -1.0
patience_counter = 0
history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}

# Default 0.5 thresholds for fast validation during training
DEFAULT_THRESHOLDS = [0.5] * config.NUM_CLASSES

# -------------------------------------------------------
# Training Loop
# -------------------------------------------------------
for epoch in range(config.EPOCHS):
    print(f"\nEpoch {epoch+1}/{config.EPOCHS}")

    # TRAIN
    model.train()
    running_loss = 0.0
    train_bar = tqdm(train_loader)

    for batch_idx, (images, labels) in enumerate(train_bar):
        images, labels = images.to(config.DEVICE), labels.to(config.DEVICE)

        optimizer.zero_grad()
        with autocast("cuda", enabled=config.USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()
    train_loss = running_loss / len(train_loader)

    # VALIDATION (Fast evaluation using default 0.5 threshold)
    model.eval()
    val_loss = 0.0
    val_probs, val_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(val_loader):
            images, labels = images.to(config.DEVICE), labels.to(config.DEVICE)
            with autocast("cuda", enabled=config.USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)

            probs = torch.sigmoid(logits)
            val_loss += loss.item()
            val_probs.append(probs.cpu().numpy())
            val_labels.append(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_probs = np.concatenate(val_probs)
    val_labels = np.concatenate(val_labels)

    # Evaluate Val Macro F1 at standard 0.5 threshold
    val_metrics = evaluate_with_thresholds(val_labels, val_probs, DEFAULT_THRESHOLDS)
    macro_f1 = val_metrics["macro_f1"]

    # History Logging
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_macro_f1"].append(macro_f1)

    print(f"Train Loss : {train_loss:.5f} | Val Loss : {val_loss:.5f} | Val Macro F1 (th=0.5): {macro_f1:.4f}")

    # Save Checkpoint based on highest Val Macro F1
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        patience_counter = 0

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "macro_f1": macro_f1,
                "epoch": epoch,
            },
            os.path.join(config.CHECKPOINT_DIR, "best_model.pth"),
        )
        print("--> Saved new best model checkpoint.")
    else:
        patience_counter += 1

    if patience_counter >= config.PATIENCE:
        print("\nEarly stopping triggered.")
        break

# Save Training History CSV
pd.DataFrame(history).to_csv(os.path.join(config.RESULTS_DIR, "training_history.csv"), index=False)

# Render & Save Training Loss / Macro F1 Curves
plot_training_history(history, config.RESULTS_DIR)


# -------------------------------------------------------
# POST-TRAINING: THRESHOLD OPTIMIZATION ON VALIDATION SET
# -------------------------------------------------------
print("\n" + "=" * 60)
print("OPTIMIZING THRESHOLDS ON VALIDATION SET (BEST MODEL)")
print("=" * 60)

# Load Best Model Checkpoint
checkpoint = torch.load(os.path.join(config.CHECKPOINT_DIR, "best_model.pth"))
model.load_state_dict(checkpoint["model"])
model.eval()

# Single validation pass to get probabilities from best model
val_probs, val_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(val_loader):
        images = images.to(config.DEVICE)
        with autocast("cuda", enabled=config.USE_AMP):
            logits = model(images)
        probs = torch.sigmoid(logits)
        val_probs.append(probs.cpu().numpy())
        val_labels.append(labels.numpy())

val_probs = np.concatenate(val_probs)
val_labels = np.concatenate(val_labels)

# Find optimal per-class thresholds ONCE
optimal_val_thresholds, optimal_val_f1s = find_best_thresholds(val_labels, val_probs)

print("\nLearned Optimal Thresholds (from Validation Set):")
for label, th, f1 in zip(config.LABEL_COLS, optimal_val_thresholds, optimal_val_f1s):
    print(f"{label:20s}: Optimal Threshold = {th:.3f} | Val F1 = {f1:.4f}")

# Save updated checkpoint with optimal thresholds attached
checkpoint["thresholds"] = optimal_val_thresholds
torch.save(checkpoint, os.path.join(config.CHECKPOINT_DIR, "best_model.pth"))
np.save(os.path.join(config.RESULTS_DIR, "best_thresholds.npy"), optimal_val_thresholds)


# -------------------------------------------------------
# FINAL HELD-OUT TEST EVALUATION & PLOTTING
# -------------------------------------------------------
print("\n" + "=" * 60)
print("RUNNING FINAL TEST EVALUATION & GENERATING ARTIFACTS")
print("=" * 60)

test_probs, test_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(config.DEVICE)
        with autocast("cuda", enabled=config.USE_AMP):
            logits = model(images)
        probs = torch.sigmoid(logits)

        test_probs.append(probs.cpu().numpy())
        test_labels.append(labels.numpy())

test_probs = np.concatenate(test_probs)
test_labels = np.concatenate(test_labels)

# 1. Compute Metrics using Validation-Learned Thresholds
test_results = evaluate_with_thresholds(test_labels, test_probs, optimal_val_thresholds, save_dir=config.RESULTS_DIR)

# 2. Display Classification Report
print(f"\nFinal Test Macro F1: {test_results['macro_f1']:.4f}\n")
print("Classification Report:")
print(test_results["report_str"])

# 3. Display PR-AUC & Threshold Table
print("\nPR-AUC & Metric Summary Table:")
print(test_results["metrics_df"].to_string(index=False))

# 4. Generate & Display All Visual Plot Artifacts
print("\nGenerating Plots...")
plot_combined_pr_curves(test_labels, test_probs, config.LABEL_COLS, config.RESULTS_DIR)
plot_f1_vs_thresholds(test_labels, test_probs, config.LABEL_COLS, optimal_val_thresholds, config.RESULTS_DIR)
plot_confusion_matrices(test_labels, test_results["predictions"], config.LABEL_COLS, config.RESULTS_DIR)

print(f"\nAll plots, tables, and reports saved directly to: {config.RESULTS_DIR}")

Overwriting /kaggle/working/train.py


In [15]:
!python /kaggle/working/train.py


Dataset Summary
----------------------------
Train : 79845
Val   : 10078
Test  : 10079
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 120MB/s]

Epoch 1/50
100%|█████████████████████████████████████████| 630/630 [02:14<00:00,  4.67it/s]
Train Loss : 0.01886 | Val Loss : 0.01605 | Val Macro F1 (th=0.5): 0.4073
--> Saved new best model checkpoint.

Epoch 2/50
100%|█████████████████████████████████████████| 630/630 [00:59<00:00, 10.54it/s]
Train Loss : 0.01472 | Val Loss : 0.01506 | Val Macro F1 (th=0.5): 0.5388
--> Saved new best model checkpoint.

Epoch 3/50
100%|█████████████████████████████████████████| 630/630 [00:57<00:00, 10.97it/s]
Train Loss : 0.01292 | Val Loss : 0.01540 | Val Macro F1 (th=0.5): 0.5135

Epoch 4/50
100%|█████████████████████████████████████████| 630/630 [00:57<00:00, 10.90it